In [6]:
from pyspark.sql import functions as F, Window as W

def clean_cols(df):
    return df.toDF(*[c.strip().lower().replace(" ", "_").replace("-", "_") for c in df.columns])

def dedupe(df, keys, order_col="ingested_at"):
    w = W.partitionBy(*keys).orderBy(F.col(order_col).desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

def write_silver(df, name):
    (df.withColumn("silver_loaded_at", F.current_timestamp())
       .write.mode("overwrite").option("overwriteSchema", "true")
       .format("delta").saveAsTable(name))
    print(name, spark.table(name).count())

bronze = lambda t: clean_cols(spark.table(f"lh_bronze.{t}"))

StatementMeta(, ef458269-5ee2-4d06-a817-d4847742984e, 8, Finished, Available, Finished, False)

In [4]:
orders = (bronze("dbo.sql_orders")
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("postal_code", F.lpad(F.col("postal_code").cast("string"), 5, "0"))
    .withColumn("region", F.initcap(F.trim("region")))
    .filter(F.col("order_id").isNotNull()))
write_silver(dedupe(orders, ["order_id"]), "orders")

lines = (bronze("dbo.sql_order_lines")
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("sales", F.col("sales").cast("decimal(12,2)"))
    .withColumn("discount", F.col("discount").cast("decimal(5,2)"))
    .withColumn("profit", F.col("profit").cast("decimal(12,2)")))
write_silver(dedupe(lines, ["row_id"]), "order_lines")

ship = (bronze("dbo.sql_shipments")
    .withColumn("ship_date", F.to_date("ship_date"))
    .withColumn("qty_shipped", F.col("qty_shipped").cast("int"))
    .withColumn("qty_ordered", F.col("qty_ordered").cast("int")))
write_silver(dedupe(ship, ["shipment_id"]), "shipments")

StatementMeta(, ef458269-5ee2-4d06-a817-d4847742984e, 6, Finished, Available, Finished, False)

orders 5009
order_lines 9994
shipments 9994


In [7]:
cust = (bronze("dbo.file_customers")
    .withColumn("customer_name", F.initcap(F.trim("customer_name")))
    .withColumn("postal_code", F.lpad(F.col("postal_code").cast("string"), 5, "0")))
write_silver(dedupe(cust, ["customer_id"]), "customers")

# Superstore re-uses some product_ids for different names: keep one row per id
prod = bronze("dbo.file_products").withColumn("product_name", F.trim("product_name"))
write_silver(dedupe(prod, ["product_id"], order_col="product_name"), "products")

inv = (bronze("dbo.file_inventory_snapshots")
    .withColumn("snapshot_date", F.to_date("snapshot_date"))
    .withColumn("qty_on_hand", F.col("qty_on_hand").cast("int"))
    .withColumn("qty_on_order", F.col("qty_on_order").cast("int"))
    .filter(F.col("qty_on_hand") >= 0))
write_silver(dedupe(inv, ["snapshot_date", "product_id", "region"]), "inventory_snapshots")

StatementMeta(, ef458269-5ee2-4d06-a817-d4847742984e, 9, Finished, Available, Finished, False)

customers 793
products 1862
inventory_snapshots 1549184
